In [4]:
from vpython import *
import math

current = 0
current = canvas.get_selected()
if current != 0:
    current.delete()
# --------------------
# Scene setup 
# --------------------
intro_scene = canvas() # set the variable intro_scene to "hold" our canvas
intro_scene.title = "Moment of Inertia" # set the title
intro_scene.width = 800 # specify the width
intro_scene.height = 600 # specify the height
intro_scene.background = color.black # set the background to the desired color
intro_scene.range= 20
# ----------------
# constants
# ----------------
g = 9.8 # in m/s
t = 0
dt = 0.001

# ----------------
# objects
# ----------------
    
# create our parent class
class Object:
    def __init__(self, mass, position, col, velocity = vector(0,0,0)):
        self.m = mass
        self.pos = position
        self.v = velocity
        self.p = velocity * mass

        self.color = col
        
        
    def collision(self, position):
        # function to detect if anything is hitting the object
        pass
    
    def grav(self):
        # function to determine force of gravity on object
        return vector(0, -self.m * g, 0)
        
    def update_v(self):
        pass
    
    def update_p(self):
        pass
    
    def update_pos(self):
        pass
 
# define Sphere object
class Sphere(Object):
    def __init__(self, rad, mass, position, col, velocity = vector(0,0,0)):
        
        super().__init__(mass, position, col, velocity)
        self.radius = rad
        self.shape = sphere(pos = position, radius = rad, color = col)
        
        self.norm = vector(0,0,0)
        self.contact_point = vector(0,0,0)

        self.interval = [position.x-rad, position.x+rad]
        
    def collision(self):
        # handle collision with ground
        if self.pos.y-self.radius <= -5:
            self.pos.y=-5+self.radius
            self.shape.y = -5+self.radius
            
            self.norm = -self.grav()
            self.contact_point = self.pos-vector(0,self.radius,0)
            
            self.v.y= 0
            self.p.y = 0
        # if sphere is not on ground
        else:
            self.norm = vector(0,0,0)
            self.contact_point = vector(0,0,0)


    def update_p(self, dt):
        F_total = self.grav() + self.norm
        self.p += F_total * dt

    def update_v(self):
        self.v = self.p/self.m

    def update_pos(self, dt):
        self.pos += self.v * dt
        self.shape.pos += self.v * dt

        self.interval[0] = self.pos.x - self.radius
        self.interval[1] = self.pos.x + self.radius


# define the Box object
class Box(Object):
    
    def __init__(self, length=1, height=1, width=1, mass = 1, position = vector(0,0,0), color = color.white, velocity = vector(0,0,0), axis = vector(0,0,0)):
        
        super().__init__(mass, position, color, velocity)
        
        self.length = length
        self.height = height
        self.width = width
        self.axis = axis
        self.shape = box(pos = position, color = color, axis = axis, length = length, height = height, width = width)

        self.norm = vector(0,0,0)
        self.contact_point = vector(0,0,0)

        self.interval = [position.x - length/2, position.x + length/2]

    def collision(self):
        if self.pos.y-self.height/2 <= -5:
            self.pos.y= -5 + self.height/2
            self.shape.y = -5 + self.height/2
            
            self.norm = -self.grav()
            self.contact_point = self.pos-vector(0,self.length/2,0)
            
            self.v.y= 0
            self.p.y = 0
        # if sphere is not on ground
        else:
            self.norm = vector(0,0,0)
            self.contact_point = vector(0,0,0)

    def update_p(self, dt):
        F_total = self.grav() + self.norm
        self.p += F_total * dt
        
    def update_v(self):
        self.v = self.p/self.m

    def update_pos(self,dt):
        self.pos += self.v * dt
        self.shape.pos += self.v * dt

        self.interval[0] = self.pos.x - self.length/2
        self.interval[1] = self.pos.x + self.length/2


# ----------------
# functions
# ----------------

def handle_1D_collision(sphere_list):
        
    # sort the list from lowest starting value to highest starting value
    sphere_list = sorted(sphere_list, key=lambda sphere: sphere.interval[0])

    # look for collisions

    for i in range(len(sphere_list)):
        sphere1 = sphere_list[i]

        for j in range(i+1, len(sphere_list)):
            sphere2 = sphere_list[j]

            # check if the left side of sphere2 is to the right of sphere1
            # AKA not overlapping
            if sphere2.interval[0] > sphere1.interval[1]:
                break
            # update velocity and momentum
            else:
                # CURRENTLY NOT ACTUAL PHYSICAL COLLISION
                temp1 = sphere1.v
                temp2 = sphere2.v
                
                sphere1.v = (sphere1.m - sphere2.m)/(sphere1.m + sphere2.m) * temp1 + (2*sphere2.m)/(sphere1.m + sphere2.m) * temp2
                sphere1.p = sphere1.v * sphere1.m

                sphere2.v = (2*sphere1.m)/(sphere1.m + sphere2.m)*temp1 + (sphere2.m - sphere1.m)/(sphere1.m + sphere2.m) * temp2
                sphere2.p = sphere2.v * sphere2.m

    
# ----------------
# creating scene
# ----------------

# create the ground
ground = box(pos=vec(0,-15,0), size=vec(100,20,1),color = color.green)
#slope = box(pos=vec(10,-15,0), size = vec(100,20.5,1), color = color.green, axis = vector(1,1,0)) #represented as the line y = x - 25
ball_list = []
ball = Sphere(0.5, 1.5, vector(8,-4.5,0), color.red, vector(-3,0,0))
ball2 = Sphere(0.5, 1, vector(0,-4.5,0), color.red, vector(2,0,0))
ball3 = Sphere(0.5, 2, vector(-4.5,-4,0), color.red, vector(3,0,0))
box = Box(length = 1, width = 1, height = 1, mass = 1, position = vector(-8,-4.5,0), color = color.red, velocity = vector(1,0,0))

ball_list.append(ball)
ball_list.append(ball2)
ball_list.append(ball3)
ball_list.append(box)


# ----------------
# start simulation
# ----------------
while t < 10:
    rate(1000)
    handle_1D_collision(ball_list)
    for i in ball_list:
        i.update_p(dt)
        i.update_v()
        i.update_pos(dt)
        i.collision()
    
    
    
    t += dt

print(ball.v)
print(ball2.v)

<IPython.core.display.Javascript object>

<4.46667, 0, 0>
<0.133333, 0, 0>


Make a setup to check collisions only, no dynamics.

Make an exercise by playing with only the objects.